Physical → Data → Knowledge → Inference → Decision

---
#### 1. Generate/update knowledge
python update_knowledge_from_csv.py

Generate **SprayLine_runtime_inferred.ttl**

---
#### 2. Send test data
python mqtt_publish_example.py


In [ ]:
import json
import paho.mqtt.client as mqtt

payload = {
    'window_id': 'w0413004',
    'timestamp': '2026-04-26T09:00:00+08:00',
    'filter': {'in_flow': 100.0, 'out_flow': 72.0},
    'nozzle': {'spray_width': 82.0, 'spray_pressure': 2.4},
    'quality': {'measured_thickness': 55.0},
    'target': {'spray_width': 80.0, 'spray_pressure': 2.5, 'thickness': 50.0}
}

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.connect('127.0.0.1', 1883, 60)
client.publish('sprayline/lineA/window', json.dumps(payload), qos=1)
client.disconnect()
print(json.dumps(payload, indent=2))


---
#### 3. Start MQTT listener
python mqtt_to_ttl_subscriber.py


In [ ]:
from __future__ import annotations
import json
from datetime import datetime, timezone
from pathlib import Path
import paho.mqtt.client as mqtt
from rdflib import Graph, Namespace, Literal, RDF
from rdflib.namespace import XSD

BASE_DIR = Path(__file__).resolve().parent
OBS_TTL = BASE_DIR / 'SprayLine_runtime_observation.ttl'
MIS = Namespace('http://nkust.edu.tw/mislab#')
STH = Namespace('http://nkust.edu.tw/mislab/cnc/ontology/sth#')

BROKER_HOST = '127.0.0.1'
BROKER_PORT = 1883
TOPIC = 'sprayline/+/window'


def json_to_observation_ttl(payload: dict) -> None:
    window_id = payload['window_id']
    w = STH[f'Window_{window_id}']
    g = Graph()
    g.bind('mis', MIS)
    g.bind('sth', STH)
    g.add((w, RDF.type, STH.SensingWindow))
    g.add((w, MIS.hasWindowId, Literal(window_id)))
    g.add((w, MIS.inFlow, Literal(payload['filter']['in_flow'], datatype=XSD.decimal)))
    g.add((w, MIS.outFlow, Literal(payload['filter']['out_flow'], datatype=XSD.decimal)))
    g.add((w, MIS.sprayWidth, Literal(payload['nozzle']['spray_width'], datatype=XSD.decimal)))
    g.add((w, MIS.targetSprayWidth, Literal(payload['target']['spray_width'], datatype=XSD.decimal)))
    g.add((w, MIS.sprayPressure, Literal(payload['nozzle']['spray_pressure'], datatype=XSD.decimal)))
    g.add((w, MIS.targetSprayPressure, Literal(payload['target']['spray_pressure'], datatype=XSD.decimal)))
    g.add((w, MIS.measuredThickness, Literal(payload['quality']['measured_thickness'], datatype=XSD.decimal)))
    g.add((w, MIS.targetThickness, Literal(payload['target']['thickness'], datatype=XSD.decimal)))
    ts = payload.get('timestamp', datetime.now(timezone.utc).isoformat())
    g.add((w, MIS.observedAt, Literal(ts, datatype=XSD.dateTime)))
    g.serialize(destination=OBS_TTL, format='turtle')


def on_connect(client, userdata, flags, reason_code, properties=None):
    print(f'Connected: {reason_code}')
    client.subscribe(TOPIC)
    print(f'Subscribed: {TOPIC}')


def on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode('utf-8'))
        json_to_observation_ttl(payload)
        print(f'Wrote observation TTL from {msg.topic}: {OBS_TTL}')
        # Optional: run RDF-native inference immediately after each MQTT message.
        from rdf_native_infer_sparql import run_inference
        run_inference()
    except Exception as exc:
        print(f'ERROR processing MQTT message: {exc}')


def main():
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
    client.loop_forever()


if __name__ == '__main__':
    main()

---
#### 4. Run inference
python rdf_native_infer_sparql.py



In [8]:
from __future__ import annotations
from pathlib import Path
from rdflib import Graph

BASE_DIR = Path.cwd()
ONTOLOGY_DIR = BASE_DIR/'ontology'
FILES = [ONTOLOGY_DIR/'SprayLine_ontology.ttl', ONTOLOGY_DIR/'SprayLine_knowledge.ttl', ONTOLOGY_DIR/'SprayLine_runtime_observation.ttl']
RULE_DIR = BASE_DIR/'rules'
OUTPUT_TTL = BASE_DIR/'SprayLine_runtime_inferred_sparql.ttl'

def load_graph() -> Graph:
    g = Graph()
    for f in FILES:
        g.parse(f, format='turtle')
    return g

def apply_construct_rule(g: Graph, rule_file: Path) -> int:
    added = 0
    for triple in g.query(rule_file.read_text(encoding='utf-8')):
        if triple not in g:
            g.add(triple)
            added += 1
    return added

def run_inference() -> Graph:
    g = load_graph()
    for rule in sorted(RULE_DIR.glob('*.rq')):
        print(f'{rule.name}: added {apply_construct_rule(g, rule)} triples')
    g.serialize(destination=OUTPUT_TTL, format='turtle')
    print(f'Saved: {OUTPUT_TTL}')
    print(f'Total triples: {len(g)}')
    return g

if __name__ == '__main__':
    run_inference()


01_filter_state.rq: added 2 triples
02_nozzle_state.rq: added 4 triples
03_process_state.rq: added 3 triples
04_diagnosis_construct.rq: added 7 triples
Saved: e:\Dropbox\Codes\Python\Ontology\SprayLine\sprayline_rdf_native_mqtt_pipeline\SprayLine_runtime_inferred_sparql.ttl
Total triples: 816


01_filter_state.rq: added triples for flowLossRatio and inferredFilterState
02_nozzle_state.rq: added triples for sprayWidthErrorRatio, pressureErrorRatio, nozzleErrorRatio, and inferredNozzleState
03_process_state.rq: added triples for thicknessErrorRatio, maxComponentSeverityRank, and process state
04_diagnosis_construct.rq: added diagnosis triples
Expected result:
  Filter state:  mis:FilterModerateBlock
  Nozzle state:  mis:NozzleEarly
  Process state: mis:IssuedProcess